# Singapore Smart City: Model Evaluation & Failure Analysis

A true Senior ML Engineer does not stop at calculating mAP on a validation set. We must perform rigorous **Holdout Test Set Evaluation** and deep **Failure Analysis** to understand *where* and *why* the model fails before deploying it to production.

### Pipeline Objectives:
1. **Strict Holdout Evaluation:** Evaluate YOLOv11s on the 15% Strict Temporal Test Set (representing unseen future data to prove no data leakage).
2. **Confusion Matrices:** Analyze misclassifications between standard vehicles (e.g., Car vs. Truck).
3. **Hard Negative Mining:** Automatically extract the top 100 worst-performing frames (highest loss) to identify systematic failures (e.g., CTE tunnel glare during night hours).
4. **Deployment Decision:** Generate a final readiness report comparing the model's metrics against the SLA (e.g., >85% mAP50-95).

In [ ]:
!pip install -q ultralytics scikit-learn seaborn matplotlib pandas

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Set aesthetic style
sns.set_theme(style="whitegrid")

In [ ]:
# 1. Load the Best Fine-Tuned Model
weights_path = '/content/runs/detect/train/weights/best.pt'
data_yaml = '/content/dataset/sg_traffic/data.yaml'

print("Loading Fine-Tuned YOLOv11s Model...")
try:
    model = YOLO(weights_path)
except FileNotFoundError:
    print(f"⚠️ {weights_path} not found! Ensure you have executed the train_yolo notebook on Kaggle/Colab first.")
    print("Falling back to base model for demonstration...")
    model = YOLO('yolov11s.pt')

# 2. Strict Holdout Test Set Evaluation
print("\n🚀 Initiating Strict Temporal Holdout Evaluation...")
metrics = model.val(data=data_yaml, split='test', conf=0.25, iou=0.6)

print("\n================ SUMMARY REPORT ================")
print(f"mAP@50:     {metrics.box.map50:.3f}")
print(f"mAP@50-95:  {metrics.box.map:.3f}")
print(f"Precision:  {metrics.box.mp:.3f}")
print(f"Recall:     {metrics.box.mr:.3f}")
print("================================================")

### Failure Analysis & Hard Negative Mining
We scan the test set to find the images where the model is least confident. These are usually due to bad lighting, heavy occlusion, or domain shift.

In [ ]:
# 3. Run Inference on Test Set and Log Confidences
test_images_dir = Path('/content/dataset/sg_traffic/images/test')
failure_logs = []

print("\n🔎 Running Failure Analysis and Hard Negative extraction...")
if test_images_dir.exists():
    image_files = list(test_images_dir.glob('*.jpg'))
    # Batch process for speed
    results = model.predict(image_files, stream=True, verbose=False)
    
    for img_path, res in zip(image_files, results):
        if len(res.boxes) == 0:
            # Total failure to detect anything
            failure_logs.append({
                'image_name': img_path.name,
                'avg_confidence': 0.0,
                'issue': 'False Negative (Zero Detections)'
            })
        else:
            avg_conf = res.boxes.conf.mean().item()
            if avg_conf < 0.4: # Low confidence threshold
                failure_logs.append({
                    'image_name': img_path.name,
                    'avg_confidence': avg_conf,
                    'issue': 'Low Confidence'
                })
                
    failure_df = pd.DataFrame(failure_logs)
    failure_df = failure_df.sort_values(by='avg_confidence')
    
    print(f"Detected {len(failure_df)} critical failure frames out of {len(image_files)} test images.")
    print("\nTop 5 Hardest Frames for Model (Send these to VLM for diagnosis or add to retrain queue):")
    print(failure_df.head())
    
    # Export to Drive for the Continuous Training pipeline
    export_path = '/content/drive/MyDrive/sg_smart_city/data/silver/hard_negatives.csv'
    try:
        failure_df.to_csv(export_path, index=False)
        print(f"\n💾 Hard Negatives exported to {export_path}")
    except:
        pass
